# PandasAI Semantic Layer Testing

Test PandasAI with your actual semantic layer from `datasets/public/`.

This notebook validates several critical issues before Slackbot integration.

In [18]:
import os
import yaml
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime

import pandasai as pai
from pandasai_litellm.litellm import LiteLLM
from pandasai import Agent
from sqlalchemy import create_engine, inspect, text

print('✓ Imports successful')

✓ Imports successful


In [19]:
# Setup
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASS = os.getenv('DB_PASS')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require'
)

try:
    with engine.connect() as conn:
        conn.execute(text('SELECT 1'))
    print('✓ PostgreSQL connected')
except Exception as e:
    print(f'✗ Connection failed: {e}')
    raise

✓ PostgreSQL connected


In [20]:
# Load semantic datasets from datasets/public/
print('\nLoading semantic layer from datasets/public/...')

semantic_datasets = {}
datasets_dir = Path('datasets/public')

for table_dir in datasets_dir.iterdir():
    if table_dir.is_dir():
        schema_file = table_dir / 'schema.yaml'
        if schema_file.exists():
            try:
                with open(schema_file) as f:
                    schema = yaml.safe_load(f)
                dataset_name = schema.get('name')
                print(f'  ✓ {dataset_name} (from {table_dir.name}/schema.yaml)')
                semantic_datasets[dataset_name] = schema
            except Exception as e:
                print(f'  ✗ Error loading {table_dir.name}: {e}')

print(f'\n✓ Loaded {len(semantic_datasets)} semantic datasets')
print(f'  Datasets: {list(semantic_datasets.keys())}')

# Load relationships.yaml from the top-level public/ directory
relationships = {}
relationships_file = datasets_dir / 'relationships.yaml'
if relationships_file.exists():
    with open(relationships_file) as f:
        relationships = yaml.safe_load(f)
    n = len(relationships.get('relationships', []))
    print(f'\n✓ Loaded relationships.yaml ({n} join paths defined)')


Loading semantic layer from datasets/public/...
  ✓ payments (from payments/schema.yaml)
  ✓ sessions (from sessions/schema.yaml)
  ✓ subscriptions (from subscriptions/schema.yaml)
  ✓ users (from users/schema.yaml)

✓ Loaded 4 semantic datasets
  Datasets: ['payments', 'sessions', 'subscriptions', 'users']

✓ Loaded relationships.yaml (4 join paths defined)


In [24]:
# Configure PandasAI
llm = LiteLLM(model='gpt-4o-mini', temperature=0, timeout=30)
pai.config.set({
    'llm': llm,
    'verbose': False,
    'enable_cache': False
})

print('✓ PandasAI configured (gpt-4o-mini)')

✓ PandasAI configured (gpt-4o-mini)


In [25]:
# Load tables directly from PostgreSQL as DataFrames
print('\nLoading tables from PostgreSQL...')

pai_datasets = {}
for table_name in semantic_datasets.keys():
    try:
        df = pd.read_sql_table(table_name, engine, schema='public')
        pai_datasets[table_name] = df
        print(f'  ✓ Loaded {table_name} ({len(df)} rows)')
    except Exception as e:
        print(f'  ⚠ Could not load {table_name}: {e}')

print(f'\n✓ {len(pai_datasets)} datasets available for querying')


Loading tables from PostgreSQL...
  ✓ Loaded payments (75624 rows)
  ✓ Loaded sessions (2391407 rows)
  ✓ Loaded subscriptions (20000 rows)
  ✓ Loaded users (20000 rows)

✓ 4 datasets available for querying


## Phase 0: Schema Validation

In [23]:
# Validate schema structure
print('='*60)
print('SCHEMA VALIDATION')
print('='*60)

inspector = inspect(engine)
db_tables = inspector.get_table_names(schema='public')

for table_name, schema in semantic_datasets.items():
    print(f'\n{table_name}:')

    # Check if table exists in PostgreSQL
    source = schema.get('source', {})
    table = source.get('table')

    if table in db_tables:
        print(f'  ✓ Table exists in PostgreSQL')
    else:
        print(f'  ✗ Table not found in PostgreSQL')

    # Check schema structure
    required_fields = ['name', 'source', 'description']
    for field in required_fields:
        if field in schema:
            print(f'  ✓ {field}: present')
        else:
            print(f'  ✗ {field}: missing')

    # Check source configuration
    source_type = source.get('type')
    if source_type == 'postgres':
        print(f'  ✓ Source type: postgres')

SCHEMA VALIDATION

payments:
  ✓ Table exists in PostgreSQL
  ✓ name: present
  ✓ source: present
  ✓ description: present
  ✓ Source type: postgres

sessions:
  ✓ Table exists in PostgreSQL
  ✓ name: present
  ✓ source: present
  ✓ description: present
  ✓ Source type: postgres

subscriptions:
  ✓ Table exists in PostgreSQL
  ✓ name: present
  ✓ source: present
  ✓ description: present
  ✓ Source type: postgres

users:
  ✓ Table exists in PostgreSQL
  ✓ name: present
  ✓ source: present
  ✓ description: present
  ✓ Source type: postgres


## Phase 1: Query Single Dataset (AGENT)

In [26]:
# Phase 1: Single-Table Queries with Execution & Timing
print('\n' + '='*60)
print('PHASE 1: SINGLE-TABLE QUERIES')
print('='*60)

# Define queries to test
test_queries = [
    {
        'table': 'users',
        'query': 'How many total users are in the database?',
        'category': 'basic_count'
    },
    {
        'table': 'users',
        'query': 'What are the top 5 countries by user count?',
        'category': 'ranking'
    },
    {
        'table': 'subscriptions',
        'query': 'How many active subscriptions do we have?',
        'category': 'filtering'
    },
    {
        'table': 'subscriptions',
        'query': 'What percentage of users have annual plans?',
        'category': 'calculation'
    },
    {
        'table': 'payments',
        'query': 'What is the total revenue across all payments?',
        'category': 'aggregation'
    },
    {
        'table': 'payments',
        'query': 'Show the distribution of payments by method',
        'category': 'grouping'
    },
    {
        'table': 'sessions',
        'query': 'What is the average session duration in minutes?',
        'category': 'aggregation'
    },
    {
        'table': 'sessions',
        'query': 'Which activity type is most common?',
        'category': 'ranking'
    },
]

results_phase1 = []

for test in test_queries:
    table_name = test['table']
    query = test['query']
    category = test['category']

    print(f"\n📊 [{category.upper()}] {query}")
    print(f"   Table: {table_name}")

    if table_name not in pai_datasets:
        print(f"   ❌ Table not found")
        continue

    try:
        # Time the execution
        start_time = time.time()

        # Create agent for this table
        agent = Agent(
            [pai_datasets[table_name]])

        # Execute the query
        response = agent.chat(query)

        elapsed_time = time.time() - start_time

        # Display result
        print(f"   ✅ Success")
        print(f"   ⏱️  Response time: {elapsed_time:.2f}s")
        print(f"   📈 Result type: {type(response).__name__}")

        # Show the actual result (first 200 chars)
        result_str = str(response)[:200]
        print(f"   📋 Result: {result_str}")

        # Log the result
        results_phase1.append({
            'query': query,
            'table': table_name,
            'category': category,
            'success': True,
            'response_time': elapsed_time,
            'result_type': type(response).__name__,
            'result_preview': result_str
        })

    except Exception as e:
        elapsed_time = time.time() - start_time
        print(f"   ❌ Failed: {str(e)[:100]}")
        print(f"   ⏱️  Time to error: {elapsed_time:.2f}s")

        results_phase1.append({
            'query': query,
            'table': table_name,
            'category': category,
            'success': False,
            'error': str(e)[:100],
            'response_time': elapsed_time
        })

# Summary statistics
successful = sum(1 for r in results_phase1 if r.get('success', False))
total = len(results_phase1)
avg_time = sum(r.get('response_time', 0) for r in results_phase1) / \
    len(results_phase1) if results_phase1 else 0

print(f"\n{'='*60}")
print(f"PHASE 1 SUMMARY")
print(f"{'='*60}")
print(f"✅ Successful: {successful}/{total}")
print(f"❌ Failed: {total - successful}/{total}")
print(f"⏱️  Average response time: {avg_time:.2f}s")
print(f"📊 Success rate: {(successful/total*100):.1f}%")


PHASE 1: SINGLE-TABLE QUERIES

📊 [BASIC_COUNT] How many total users are in the database?
   Table: users
   ✅ Success
   ⏱️  Response time: 2.94s
   📈 Result type: NumberResponse
   📋 Result: 20000

📊 [RANKING] What are the top 5 countries by user count?
   Table: users
   ✅ Success
   ⏱️  Response time: 3.85s
   📈 Result type: DataFrameResponse
   📋 Result:   country  user_count
0      US        8079
1      EU        3989
2   India        3969
3    Rest        3963

📊 [FILTERING] How many active subscriptions do we have?
   Table: subscriptions
   ✅ Success
   ⏱️  Response time: 3.43s
   📈 Result type: NumberResponse
   📋 Result: 18684

📊 [CALCULATION] What percentage of users have annual plans?
   Table: subscriptions
   ✅ Success
   ⏱️  Response time: 6.43s
   📈 Result type: NumberResponse
   📋 Result: 9.87

📊 [AGGREGATION] What is the total revenue across all payments?
   Table: payments
   ✅ Success
   ⏱️  Response time: 3.58s
   📈 Result type: NumberResponse
   📋 Result: 1030560

Phase 1: Validation of LLM using SQL interregation of database: 

In [ ]:
# PHASE 1: DIRECT POSTGRESQL VALIDATION
# Compare PandasAI results against hand-written SQL queries
print('\n' + '='*80)
print('PHASE 1 DIRECT SQL VALIDATION')
print('='*80)

# Define the same queries with their SQL equivalents
validation_queries = [
    {
        'category': 'basic_count',
        'natural_language': 'How many total users are in the database?',
        'table': 'users',
        'sql': 'SELECT COUNT(*) as total_users FROM users;',
        'description': 'Count all users'
    },
    {
        'category': 'ranking',
        'natural_language': 'What are the top 5 countries by user count?',
        'table': 'users',
        'sql': '''SELECT country, COUNT(*) as user_count 
                  FROM users 
                  GROUP BY country 
                  ORDER BY user_count DESC 
                  LIMIT 5;''',
        'description': 'Top 5 countries by user count'
    },
    {
        'category': 'filtering',
        'natural_language': 'How many active subscriptions do we have?',
        'table': 'subscriptions',
        'sql': '''SELECT COUNT(*) as active_subscriptions 
                  FROM subscriptions 
                  WHERE status = 'active';''',
        'description': 'Count of active subscriptions'
    },
    {
        'category': 'calculation',
        'natural_language': 'What percentage of users have annual plans?',
        'table': 'subscriptions',
        'sql': '''SELECT 
                    ROUND(
                        COUNT(DISTINCT CASE WHEN plan = 'annual' THEN user_id END) * 100.0 / 
                        COUNT(DISTINCT user_id), 
                        2
                    ) as annual_plan_percentage
                  FROM subscriptions
                  WHERE status = 'active';''',
        'description': 'Percentage of users with annual plans'
    },
    {
        'category': 'aggregation',
        'natural_language': 'What is the total revenue across all payments?',
        'table': 'payments',
        'sql': '''SELECT SUM(amount_usd) as total_revenue 
                  FROM payments;''',
        'description': 'Total revenue from all payments'
    },
    {
        'category': 'grouping',
        'natural_language': 'Show the distribution of payments by method',
        'table': 'payments',
        'sql': '''SELECT method, COUNT(*) as payment_count, SUM(amount_usd) as total_by_method
                  FROM payments 
                  GROUP BY method 
                  ORDER BY payment_count DESC;''',
        'description': 'Payment distribution by method'
    },
    {
        'category': 'aggregation',
        'natural_language': 'What is the average session duration in minutes?',
        'table': 'sessions',
        'sql': '''SELECT ROUND(AVG(duration_minutes)::numeric, 2) as avg_session_duration
                  FROM sessions;''',
        'description': 'Average session duration'
    },
    {
        'category': 'ranking',
        'natural_language': 'Which activity type is most common?',
        'table': 'sessions',
        'sql': '''SELECT activity_type, COUNT(*) as activity_count
                  FROM sessions 
                  GROUP BY activity_type 
                  ORDER BY activity_count DESC 
                  LIMIT 1;''',
        'description': 'Most common activity type'
    },
]

results_direct_sql = []

for i, query_spec in enumerate(validation_queries, 1):
    category = query_spec['category']
    natural_language = query_spec['natural_language']
    sql = query_spec['sql']
    description = query_spec['description']

    print(f"\n{'─'*80}")
    print(f"Query {i}: [{category.upper()}]")
    print(f"{'─'*80}")
    print(f"📝 Natural Language: {natural_language}")
    print(f"📊 Description: {description}")
    print(f"\n🔍 SQL:")
    print(f"   {sql}")

    try:
        # Execute the query
        start_time = time.time()

        with engine.connect() as conn:
            result = conn.execute(text(sql))
            df = pd.DataFrame(result.fetchall(), columns=result.keys())

        elapsed_time = time.time() - start_time

        print(f"\n✅ Query executed successfully")
        print(f"⏱️  Response time: {elapsed_time:.3f}s")
        print(f"📈 Result rows: {len(df)}")
        print(f"📋 Result:\n{df.to_string(index=False)}")

        results_direct_sql.append({
            'query_num': i,
            'category': category,
            'natural_language': natural_language,
            'sql': sql,
            'success': True,
            'response_time': elapsed_time,
            'result_rows': len(df),
            'result_df': df
        })

    except Exception as e:
        elapsed_time = time.time() - start_time
        error_msg = str(e)
        print(f"\n❌ Query failed")
        print(f"⏱️  Time to error: {elapsed_time:.3f}s")
        print(f"❌ Error: {error_msg[:200]}")

        results_direct_sql.append({
            'query_num': i,
            'category': category,
            'natural_language': natural_language,
            'sql': sql,
            'success': False,
            'error': error_msg,
            'response_time': elapsed_time
        })

# Summary statistics
print(f"\n{'='*80}")
print("DIRECT SQL SUMMARY")
print(f"{'='*80}")

successful = sum(1 for r in results_direct_sql if r['success'])
total = len(results_direct_sql)

print(f"\n✅ Successful: {successful}/{total}")
print(f"❌ Failed: {total - successful}/{total}")

if successful > 0:
    successful_times = [r['response_time']
                        for r in results_direct_sql if r['success']]
    avg_time = sum(successful_times) / len(successful_times)
    min_time = min(successful_times)
    max_time = max(successful_times)

    print(f"\n⏱️  Response Times:")
    print(f"   Average: {avg_time:.3f}s")
    print(f"   Min: {min_time:.3f}s")
    print(f"   Max: {max_time:.3f}s")

print(f"📊 Success rate: {(successful/total*100):.1f}%")

# Close connection
engine.dispose()

print("\n✅ Direct SQL validation complete!")


PHASE 1 DIRECT SQL VALIDATION

────────────────────────────────────────────────────────────────────────────────
Query 1: [BASIC_COUNT]
────────────────────────────────────────────────────────────────────────────────
📝 Natural Language: How many total users are in the database?
📊 Description: Count all users

🔍 SQL:
   SELECT COUNT(*) as total_users FROM users;

✅ Query executed successfully
⏱️  Response time: 0.557s
📈 Result rows: 1
📋 Result:
 total_users
       20000

────────────────────────────────────────────────────────────────────────────────
Query 2: [RANKING]
────────────────────────────────────────────────────────────────────────────────
📝 Natural Language: What are the top 5 countries by user count?
📊 Description: Top 5 countries by user count

🔍 SQL:
   SELECT country, COUNT(*) as user_count 
                  FROM users 
                  GROUP BY country 
                  ORDER BY user_count DESC 
                  LIMIT 5;

✅ Query executed successfully
⏱️  Response tim

There is a small difference of 10.09-9.87=0.22 (2% error) in answer on query 4: what percentage of users have annual plans. 

In [34]:
agent = Agent([pai_datasets['subscriptions']])
response = agent.chat("What percentage of users have annual plans?")

print(f"Answer: {response}\n")
print("="*80)
print("PANDASAI GENERATED PYTHON CODE:")
print("="*80)

# Extract the generated code
generated_code = agent._state.last_code_generated

print(generated_code)

Answer: 9.87

PANDASAI GENERATED PYTHON CODE:
import pandas as pd
sql_query = """
SELECT 
    (COUNT(CASE WHEN plan = 'annual' THEN 1 END) * 100.0 / COUNT(*)) AS percentage_annual_users
FROM 
    table_522091d699ef89e3ad5fcb2132b15f9d
"""
result_df = execute_sql_query(sql_query)
percentage_annual_users = result_df['percentage_annual_users'].iloc[0]
result = {'type': 'number', 'value': percentage_annual_users}


In [ ]:
# PHRASING CHANGES TO FORCE PANDASAI TO USE ACTIVE USERS
# Option A: Be more specific about "active"
response = agent.chat(
    "What percentage of users with active subscriptions have an annual plan?"
)
print(f"Answer A: {response}")

# Option B: Be explicit about the calculation
response = agent.chat(
    "Count the number of distinct users with annual plans. "
    "Divide by the total number of distinct users with active subscriptions. "
    "Express as a percentage."
)
print(f"Answer B: {response}")

# Option C: Show the semantic context
response = agent.chat(
    "Among active subscriptions, what percentage are annual plans?"
)
print(f"Answer C: {response}")

Answer A: 10.094198244487261
Answer B: 10.094198244487261
Answer C: 10.094198244487261


These three responses agree with the answer obtained using SQL directly on the database, so posing the question via pandasai using specific language targeting *active* users solves the problem.  

OR
  
we could change the semantic layer in the users table to specify:
name: status
    type: string
    description: "Subscription status - ALWAYS filter by status='active' when calculating active subscription metrics"
    values: ["active", "expired", "canceled"]
    context: "Critical: Only active subscriptions should be counted in analysis queries"

## Phase 2: Multi-Dataset Queries (Agent)

Test queries that require joins across tables.

In [ ]:
# PHASE 2: MULTI-TABLE QUERIES (JOIN TESTING)
# Test PandasAI with cross-table joins using semantic layer relationships

import csv
print('\n' + '='*80)
print('PHASE 2: MULTI-TABLE QUERIES WITH JOINS')
print('='*80)

# ─────────────────────────────────────────────────────────────────────────────
# Setup: Initialize multi-dataset Agent
# ─────────────────────────────────────────────────────────────────────────────

print('\n📊 Initializing multi-table Agent...\n')

# Build relationship context from relationships.yaml
relationship_desc = ""
if relationships:
    lines = ["TABLE RELATIONSHIPS FOR JOINS:"]
    for rel in relationships.get('relationships', []):
        if 'through_table' in rel:
            lines.append(
                f"  • {rel['from_table']}.{rel['from_column']} → "
                f"{rel['through_table']}.{rel['through_column']} → "
                f"{rel['to_table']}.{rel['to_column']}"
            )
        else:
            lines.append(
                f"  • {rel['from_table']}.{rel['from_column']} ↔ "
                f"{rel['to_table']}.{rel['to_column']}"
            )
    relationship_desc = "\n".join(lines)
    print(relationship_desc)
    print()

# Configure PandasAI
pai.config.set({
    "llm": llm,
    "verbose": False,
    "enable_cache": False
})

# Create multi-table Agent
agent = Agent(
    list(pai_datasets.values()),
    description=relationship_desc or "Multi-table database with users, subscriptions, sessions, payments"
)

print(f'✅ Agent initialized with {len(pai_datasets)} datasets')
print(f'   Tables: {", ".join(pai_datasets.keys())}')

# ─────────────────────────────────────────────────────────────────────────────
# Phase 2 Queries: Multi-Table Joins
# ─────────────────────────────────────────────────────────────────────────────

print('\n' + '='*80)
print('PHASE 2: EXECUTING MULTI-TABLE QUERIES')
print('='*80)

multi_table_queries = [
    {
        'num': 1,
        'complexity': 'EASY',
        'category': 'user_subscription_join',
        'natural_language': 'How many users have active subscriptions?',
        'tables_involved': ['users', 'subscriptions'],
        'expected_difficulty': 'Simple join: users → subscriptions with status filter',
        'sql_hint': 'JOIN users ON subscriptions.user_id = users.user_id WHERE status = active'
    },
    {
        'num': 2,
        'complexity': 'EASY',
        'category': 'subscription_revenue',
        'natural_language': 'What is the total revenue from active subscriptions by country?',
        'tables_involved': ['users', 'subscriptions', 'payments'],
        'expected_difficulty': 'Multi-hop join: subscriptions → payments → users with grouping',
        'sql_hint': 'JOIN payments ON subscriptions.subscription_id; JOIN users; GROUP BY country'
    },
    {
        'num': 3,
        'complexity': 'MEDIUM',
        'category': 'user_engagement',
        'natural_language': 'Show users with annual plans and their total session count',
        'tables_involved': ['users', 'subscriptions', 'sessions'],
        'expected_difficulty': 'Join 3 tables: users → subscriptions + sessions with aggregation',
        'sql_hint': 'GROUP BY user; COUNT sessions; FILTER plan = annual'
    },
    {
        'num': 4,
        'complexity': 'MEDIUM',
        'category': 'payment_method_analysis',
        'natural_language': 'Which payment methods are most popular by user country?',
        'tables_involved': ['users', 'subscriptions', 'payments'],
        'expected_difficulty': 'Join through subscriptions: payments → subscriptions → users',
        'sql_hint': 'GROUP BY method, country; COUNT; ORDER BY count DESC'
    },
    {
        'num': 5,
        'complexity': 'MEDIUM',
        'category': 'subscription_activity',
        'natural_language': 'For each subscription plan, show average session duration and total revenue',
        'tables_involved': ['subscriptions', 'sessions', 'payments'],
        'expected_difficulty': 'Join 3 tables with multiple aggregations',
        'sql_hint': 'GROUP BY plan; AVG(duration); SUM(amount)'
    },
    {
        'num': 6,
        'complexity': 'HARD',
        'category': 'user_value_analysis',
        'natural_language': 'List users who have annual plans, high session activity (>100 sessions), and have made payments',
        'tables_involved': ['users', 'subscriptions', 'sessions', 'payments'],
        'expected_difficulty': 'Join 4 tables with multiple filters and aggregation',
        'sql_hint': 'HAVING COUNT(sessions) > 100; status = active; payments exist'
    },
    {
        'num': 7,
        'complexity': 'HARD',
        'category': 'revenue_by_engagement',
        'natural_language': 'Show average revenue per user by subscription plan and session activity level',
        'tables_involved': ['users', 'subscriptions', 'sessions', 'payments'],
        'expected_difficulty': 'Complex aggregation: sessions per user + revenue + plan grouping',
        'sql_hint': 'GROUP BY plan; AVG(revenue); COUNT(DISTINCT sessions per user)'
    },
    {
        'num': 8,
        'complexity': 'HARD',
        'category': 'cohort_analysis',
        'natural_language': 'Compare the average revenue for users who have both sessions and payments vs those who only have sessions',
        'tables_involved': ['users', 'subscriptions', 'sessions', 'payments'],
        'expected_difficulty': 'Conditional aggregation: CASE WHEN payments exist THEN revenue ELSE 0',
        'sql_hint': 'Compare: users WITH payments vs users WITHOUT payments'
    },
]

results_phase2 = []

# Execute each query
for query_spec in multi_table_queries:
    num = query_spec['num']
    complexity = query_spec['complexity']
    category = query_spec['category']
    natural_language = query_spec['natural_language']
    tables = ', '.join(query_spec['tables_involved'])
    difficulty = query_spec['expected_difficulty']

    print(f"\n{'─'*80}")
    print(f"Query {num}: [{complexity}] {category.upper()}")
    print(f"{'─'*80}")

    print(f"📝 Question: {natural_language}")
    print(f"📊 Tables: {tables}")
    print(f"🎯 Expected Difficulty: {difficulty}")

    try:
        # Time the execution
        start_time = time.time()

        # Execute the query
        response = agent.chat(natural_language)

        elapsed_time = time.time() - start_time

        # Display result
        print(f"\n✅ SUCCESS ({elapsed_time:.2f}s)")
        print(f"📈 Result type: {type(response).__name__}")

        # Show the result
        result_str = str(response)[:300]
        print(f"📋 Result:")
        print(f"   {result_str}")

        # Try to access the generated code
        try:
            generated_code = agent._state.last_code_generated
            print(f"\n🔍 Generated Code (first 200 chars):")
            print(f"   {generated_code[:200]}...")
        except:
            pass

        # Log the result
        results_phase2.append({
            'num': num,
            'category': category,
            'complexity': complexity,
            'natural_language': natural_language,
            'tables': tables,
            'success': True,
            'response_time': elapsed_time,
            'result_type': type(response).__name__,
            'result_preview': result_str
        })

    except Exception as e:
        elapsed_time = time.time() - start_time
        error_msg = str(e)[:150]

        print(f"\n❌ FAILED ({elapsed_time:.2f}s)")
        print(f"⚠️  Error: {error_msg}")

        results_phase2.append({
            'num': num,
            'category': category,
            'complexity': complexity,
            'natural_language': natural_language,
            'tables': tables,
            'success': False,
            'error': error_msg,
            'response_time': elapsed_time
        })

# ─────────────────────────────────────────────────────────────────────────────
# Phase 2 Summary
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'='*80}")
print('PHASE 2 SUMMARY')
print(f"{'='*80}\n")

successful = sum(1 for r in results_phase2 if r.get('success', False))
total = len(results_phase2)

print(f"✅ Successful: {successful}/{total}")
print(f"❌ Failed: {total - successful}/{total}")

if successful > 0:
    successful_times = [r['response_time']
                        for r in results_phase2 if r.get('success')]
    avg_time = sum(successful_times) / len(successful_times)
    min_time = min(successful_times)
    max_time = max(successful_times)

    print(f"\n⏱️  Response Times (successful queries):")
    print(f"   Average: {avg_time:.2f}s")
    print(f"   Min: {min_time:.2f}s")
    print(f"   Max: {max_time:.2f}s")

# Success by complexity
easy_queries = [r for r in results_phase2 if r['complexity'] == 'EASY']
medium_queries = [r for r in results_phase2 if r['complexity'] == 'MEDIUM']
hard_queries = [r for r in results_phase2 if r['complexity'] == 'HARD']

if easy_queries:
    easy_success = sum(1 for r in easy_queries if r.get('success'))
    print(f"\n📊 By Complexity:")
    print(f"   EASY:   {easy_success}/{len(easy_queries)} ✓")

if medium_queries:
    medium_success = sum(1 for r in medium_queries if r.get('success'))
    print(f"   MEDIUM: {medium_success}/{len(medium_queries)} ✓")

if hard_queries:
    hard_success = sum(1 for r in hard_queries if r.get('success'))
    print(f"   HARD:   {hard_success}/{len(hard_queries)} ✓")

print(f"\n📊 Overall Success Rate: {(successful/total*100):.1f}%")

# ─────────────────────────────────────────────────────────────────────────────
# Key Observations
# ─────────────────────────────────────────────────────────────────────────────

print(f"\n{'='*80}")
print('OBSERVATIONS & ANALYSIS')
print(f"{'='*80}\n")

if successful == total:
    print("✅ All multi-table queries succeeded!")
    print("   → PandasAI effectively handles complex joins")
    print("   → Semantic layer relationships are well understood")
elif successful >= total * 0.75:
    print("✅ Most queries succeeded (≥75%)")
    print("   → Core join functionality works well")
    print("   → Some edge cases may need refinement")
else:
    print("⚠️  Mixed results on multi-table queries")
    failed = [r for r in results_phase2 if not r.get('success')]
    print("   Failed queries:")
    for r in failed:
        print(
            f"     • {r['category']}: {r.get('error', 'Unknown error')[:80]}")

# Export results to CSV

print(f"\n" + "="*80)
print('EXPORTING RESULTS')
print(f"="*80)

with open('phase2_results.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=[
        'query_num', 'complexity', 'category', 'tables_involved',
        'success', 'response_time_s', 'result_type', 'error'
    ])
    writer.writeheader()
    for result in results_phase2:
        writer.writerow({
            'query_num': result['num'],
            'complexity': result['complexity'],
            'category': result['category'],
            'tables_involved': result['tables'],
            'success': result.get('success', False),
            'response_time_s': f"{result['response_time']:.2f}",
            'result_type': result.get('result_type', ''),
            'error': result.get('error', '')
        })

print(f"✅ Results exported to: phase2_results.csv")

print(f"\n{'='*80}")
print('✅ PHASE 2 COMPLETE')
print(f"{'='*80}\n")


PHASE 2: MULTI-TABLE QUERIES WITH JOINS

📊 Initializing multi-table Agent...

TABLE RELATIONSHIPS FOR JOINS:
  • subscriptions.user_id ↔ users.user_id
  • sessions.user_id ↔ users.user_id
  • payments.subscription_id ↔ subscriptions.subscription_id
  • payments.subscription_id → subscriptions.subscription_id → users.user_id

✅ Agent initialized with 4 datasets
   Tables: payments, sessions, subscriptions, users

PHASE 2: EXECUTING MULTI-TABLE QUERIES

────────────────────────────────────────────────────────────────────────────────
Query 1: [EASY] USER_SUBSCRIPTION_JOIN
────────────────────────────────────────────────────────────────────────────────
📝 Question: How many users have active subscriptions?
📊 Tables: users, subscriptions
🎯 Expected Difficulty: Simple join: users → subscriptions with status filter

✅ SUCCESS (3.47s)
📈 Result type: NumberResponse
📋 Result:
   18684

🔍 Generated Code (first 200 chars):
   import pandas as pd
sql_query = """
SELECT COUNT(DISTINCT user_id) AS ac

## Summary

In [10]:
print('\n' + '='*60)
print('TESTING SUMMARY')
print('='*60)
print(f'\nSemantic datasets loaded: {len(semantic_datasets)}')
print(f'PandasAI datasets ready: {len(pai_datasets)}')
print(f'\nNext steps:')
print('1. Review TESTING_GUIDE.md for detailed 9-phase testing')
print('2. Run each phase to validate PandasAI with your semantic layer')
print('3. Check CSV outputs for detailed metrics')
print('\n✓ Setup validation complete!')


TESTING SUMMARY

Semantic datasets loaded: 4
PandasAI datasets ready: 4

Next steps:
1. Review TESTING_GUIDE.md for detailed 9-phase testing
2. Run each phase to validate PandasAI with your semantic layer
3. Check CSV outputs for detailed metrics

✓ Setup validation complete!
